In [ ]:
!pip install git+https://github.com/openai/whisper.git
!pip install git+https://github.com/snakers4/silero-vad.git
!sudo apt-get install -y ffmpeg

  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-trwmhyb4
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-trwmhyb4
  Resolved https://github.com/openai/whisper.git to commit c0d2f624c09dc18e709e37c2ad90c039a4eb72a2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=0eeee81110e7cf8c1110f3778113a371f1fe29986775edc8c973c3283cd912c1
  Stored in directory: /tmp/pip-ephem-wheel-cache-r0sg3uh4/wheels/c3/03/25/5e0ba78bc27a3a089f137c9f1d92fdfce16d06996c071a016c
Successfully built openai-whisper
  Cloning https://github.com/snakers4/silero-vad.git to /tmp/pip-req-build-ad71tefr
  Running command git clone --filter=blob:none --quiet https://github.com/snakers4/silero-vad.git /tmp/pip-req-build-ad71tefr
  Resolved https://github

## only one output

In [ ]:
import whisper
import torch
import os
import time

def format_timestamp(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    milliseconds = int((seconds % 1) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{milliseconds:03d}"

def save_srt(result, output_file):
    """Save transcription result as SRT subtitle file"""
    with open(output_file, 'w', encoding='utf-8') as f:
        for i, segment in enumerate(result["segments"], 1):
            start_time = format_timestamp(segment["start"])
            end_time = format_timestamp(segment["end"])
            text = segment["text"].strip()

            f.write(f"{i}\n")
            f.write(f"{start_time} --> {end_time}\n")
            f.write(f"{text}\n\n")

def transcribe_media(media_path, model_size="large-v2", language="es"):
    """
    Transcribe a single media (audio or video) file
    """
    print("=" * 50)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading Whisper model: {model_size} on {device}...")
    model = whisper.load_model(model_size).to(device)

    print(f"Transcribing {media_path}...")
    t0 = time.time()
    result = model.transcribe(media_path, language=language)
    print(f"Transcription finished in {time.time() - t0:.2f}s")

    # Save as SRT file
    base, _ = os.path.splitext(media_path)
    srt_output_file = f"{base}.srt"
    save_srt(result, srt_output_file)
    print(f"Saved subtitles -> {srt_output_file}")

    return result

# Example usage in Colab
media_file = "/content/LA ERA DEL HIELO 1_first_10min.mp4"
result = transcribe_media(media_file, model_size="large-v2", language="es")

Loading Whisper model: large-v2 on cuda...


100%|█████████████████████████████████████| 2.87G/2.87G [00:38<00:00, 79.9MiB/s]


Transcribing /content/LA ERA DEL HIELO 1_first_10min.mp4...
Transcription finished in 101.44s
Saved subtitles -> /content/LA ERA DEL HIELO 1_first_10min.srt


## Whole directory and change parameters

In [ ]:
# Configuration Cell - Run this first
model_size = "large-v2"  # Options: tiny, base, small, medium, large, large-v2, large-v3
language = "fr"          # Options: en, es, fr, de, it, pt, nl, pl, ja, zh, etc.
input_paths = ["/content/folder1"]  # List of files or folders ["/content/audio_file.mp3", "/content/videos/"]

In [ ]:
import whisper
import torch
import time
import os

def transcribe_media(media_path, model, language):
    """
    Transcribe a single media file using the pre-loaded Whisper model.
    """
    try:
        print(f"Transcribing: {media_path}...")
        transcribe_start = time.time()

        # Transcribe the audio
        result = model.transcribe(media_path, language=language)

        transcribe_time = time.time() - transcribe_start

        # Generate SRT file path
        base, _ = os.path.splitext(media_path)
        srt_output_file = f"{base}.srt"

        # Save as SRT file
        save_srt(result, srt_output_file)

        print(f"Successfully transcribed '{media_path}' -> '{srt_output_file}' in {transcribe_time:.2f}s")

    except Exception as e:
        print(f"Error transcribing '{media_path}': {e}")

def format_timestamp(seconds):
    """Convert seconds to SRT timestamp format (HH:MM:SS,mmm)"""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    milliseconds = int((seconds % 1) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{milliseconds:03d}"

def save_srt(result, output_file):
    """Save transcription result as SRT subtitle file"""
    with open(output_file, 'w', encoding='utf-8') as f:
        for i, segment in enumerate(result["segments"], 1):
            start_time = format_timestamp(segment["start"])
            end_time = format_timestamp(segment["end"])
            text = segment["text"].strip()

            f.write(f"{i}\n")
            f.write(f"{start_time} --> {end_time}\n")
            f.write(f"{text}\n\n")

# Use configuration variables instead of argparse
# model_size, language, and input_paths should be defined in previous cell

# Supported media file extensions
supported_extensions = ['.mp3', '.mp4', '.m4a', '.wav', '.flac', '.mkv', '.avi', '.mov']

# Collect all files to transcribe
files_to_transcribe = []
for path in input_paths:
    if os.path.isfile(path):
        if os.path.splitext(path)[1].lower() in supported_extensions:
            files_to_transcribe.append(path)
        else:
            print(f"Warning: Skipping unsupported file type: {path}")
    elif os.path.isdir(path):
        print(f"Scanning folder: {path}")
        for root, _, files in os.walk(path):
            for file in files:
                if os.path.splitext(file)[1].lower() in supported_extensions:
                    full_path = os.path.join(root, file)
                    files_to_transcribe.append(full_path)
    else:
        print(f"Error: Path not found or is not a file/directory: {path}")

if not files_to_transcribe:
    print("No supported media files found to transcribe.")
else:
    # Check for GPU
    if torch.cuda.is_available():
        print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("Using CPU")

    # Load the Whisper model once
    print(f"Loading Whisper model: {model_size}...")
    start_time = time.time()
    try:
        model = whisper.load_model(model_size)
        load_time = time.time() - start_time
        print(f"Model loaded in {load_time:.2f} seconds")
    except Exception as e:
        print(f"Error loading model: {e}")
        raise

    print("=" * 50)
    print(f"Found {len(files_to_transcribe)} files to transcribe.")

    for file_path in files_to_transcribe:
        transcribe_media(file_path, model, language)
        print("-" * 50)

Scanning folder: /content/folder1
Using GPU: Tesla T4
Loading Whisper model: large-v2...


100%|█████████████████████████████████████| 2.87G/2.87G [01:27<00:00, 35.3MiB/s]


Model loaded in 118.47 seconds
Found 2 files to transcribe.
Transcribing: /content/folder1/French III - Lesson 02.mp3...
Successfully transcribed '/content/folder1/French III - Lesson 02.mp3' -> '/content/folder1/French III - Lesson 02.srt' in 134.61s
--------------------------------------------------
Transcribing: /content/folder1/French III - Lesson 01.mp3...
Successfully transcribed '/content/folder1/French III - Lesson 01.mp3' -> '/content/folder1/French III - Lesson 01.srt' in 140.72s
--------------------------------------------------
